# Palm Passport 01: where is the oil palm?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aldinorizaldy/geomata/blob/palm-passport-step2/notebooks/01_palm_mask.ipynb)

**Area:** Siak district, Riau, starting with one 20 x 20 km box.
**What you get:** a map of industrial and smallholder oil palm, hectares of each,
and a first deforestation-free (EUDR) check on every palm pixel.

**Before you start (one time only):** you need a Google Cloud project that is
registered for Earth Engine. Go to https://code.earthengine.google.com/register,
sign in with your Google account, create or pick a project, and copy its ID
into `PROJECT_ID` below. Then use **Runtime > Run all**. Colab will ask you to
sign in to Google once.

Data: Descals et al. global oil palm map 2019 (`BIOPAMA/GlobalOilPalm/v1`),
JRC forest cover 2020, Hansen Global Forest Change, Sentinel-2.

In [ ]:
# @title 0. Settings: fill in your Earth Engine project ID, then run all cells
PROJECT_ID = "your-ee-project-id"  # @param {type:"string"}
REPO_REF = "palm-passport-step2"  # @param {type:"string"}
BOX_KM = 20  # @param {type:"integer"}
# Leave both at 0 to auto-pick the 20 km box in Siak with the most smallholder palm.
CENTER_LON = 0.0  # @param {type:"number"}
CENTER_LAT = 0.0  # @param {type:"number"}

In [ ]:
# @title 1. Install Palm Passport and sign in to Earth Engine
%pip install -q "palm-passport @ git+https://github.com/aldinorizaldy/geomata.git@$REPO_REF" geemap

import ee
import geemap.foliumap as geemap
from palm_passport import aoi, assets, auth, palm_mask

auth.init(PROJECT_ID)

In [ ]:
# @title 2. Study area: Siak district and one 20 km test box
siak = aoi.district("Siak", "Riau")
if CENTER_LON and CENTER_LAT:
    study_box = aoi.box(CENTER_LON, CENTER_LAT, BOX_KM)
    print(f"Using your box centred on {CENTER_LON:.4f}, {CENTER_LAT:.4f}")
else:
    best = aoi.densest_smallholder_box(siak, BOX_KM).getInfo()
    props = best["properties"]
    study_box = ee.Geometry(best["geometry"])
    CENTER_LON, CENTER_LAT = props["center_lon"], props["center_lat"]
    print(f"Auto-picked box centred on {CENTER_LON:.4f}, {CENTER_LAT:.4f} "
          f"({props['smallholder_frac']:.0%} smallholder palm)")
    print("Tip: copy these numbers into CENTER_LON / CENTER_LAT so notebook 02 uses the same box.")
print(f"Siak district area: {siak.geometry().area(100).divide(1e6).getInfo():,.0f} km2")

In [ ]:
# @title 3. How much palm is there?
def show_area(name, region, scale):
    ha = palm_mask.area_by_class_ha(region, scale).getInfo()
    ind, sh = ha.get("industrial_ha", 0), ha.get("smallholder_ha", 0)
    total = ind + sh
    share = sh / total if total else 0
    print(f"{name}: {total:,.0f} ha palm  |  industrial {ind:,.0f} ha  |  "
          f"smallholder {sh:,.0f} ha ({share:.0%})")

show_area("Test box", study_box, 10)
show_area("Siak district", siak.geometry(), 30)  # coarser scale keeps it fast

In [ ]:
# @title 4. EUDR first pass: palm on 2020 forest, or tree-cover loss after 2020
flags = palm_mask.deforestation_flags(after_year=2020)
print("Using", assets.resolve("jrc_gfc2020"), "and", assets.resolve("hansen"))
area = ee.Image.pixelArea().divide(1e4)
flag_ha = flags.multiply(area).reduceRegion(
    ee.Reducer.sum(), study_box, scale=10, maxPixels=1e10, tileScale=4
).getInfo()
print(f"Palm on land mapped as forest in 2020: {flag_ha['forest2020']:,.0f} ha")
print(f"Palm with tree-cover loss after 2020:  {flag_ha['loss_after']:,.0f} ha")
print(f"Total flagged for review:              {flag_ha['flag']:,.0f} ha")

In [ ]:
# @title 5. Map
classes = palm_mask.oil_palm_classes()
rgb = assets.sentinel2_rgb(study_box, "2024-01-01", "2025-12-31")

m = geemap.Map(center=[CENTER_LAT, CENTER_LON], zoom=12)
m.add_basemap("SATELLITE")
m.addLayer(rgb, {"min": 0, "max": 0.3}, "Sentinel-2 2024-25 (cloud-free median)")
m.addLayer(classes.clip(siak), {"min": 1, "max": 2,
           "palette": [palm_mask.PALETTE["industrial"], palm_mask.PALETTE["smallholder"]]},
           "Oil palm (2019)")
m.addLayer(flags.select("flag").selfMask().clip(study_box), {"palette": ["#ff00ff"]},
           "EUDR flag (review)")
m.addLayer(ee.Image().paint(siak, 0, 2), {"palette": ["white"]}, "Siak district")
m.addLayer(ee.Image().paint(ee.FeatureCollection([ee.Feature(study_box)]), 0, 3),
           {"palette": ["yellow"]}, "Test box")
m.add_legend(title="Legend", legend_dict={
    "Industrial palm": palm_mask.PALETTE["industrial"][1:],
    "Smallholder palm": palm_mask.PALETTE["smallholder"][1:],
    "EUDR flag": "ff00ff",
})
m

In [ ]:
# @title 6. (Optional) Save the palm mask for the box to your Google Drive
EXPORT = False  # @param {type:"boolean"}
if EXPORT:
    task = ee.batch.Export.image.toDrive(
        image=classes.unmask(0).clip(study_box).byte(),
        description="palm_passport_siak_palm_mask",
        folder="palm_passport", region=study_box, scale=10, crs=aoi.UTM_47N,
        maxPixels=1e10)
    task.start()
    print("Export started. Watch it at https://code.earthengine.google.com/tasks")

## What to look at

- Zoom into the yellow box. Do the green (smallholder) and orange (industrial) areas match what you see on the satellite image?
- The pink pixels are palm that the EUDR check would ask questions about. Most of the "forest in 2020" ones are likely disagreements between two maps; "loss after 2020" is the stronger warning.
- Next: **notebook 02** dates when each palm pixel was planted.